# 추출 오류의 원인을 찾고 개선 전후를 비교합니다

**이 교안의 목표는 무엇이 잘못됐는지 조사하고, 수정 결과를 채택할지 판단하는 것입니다.**

01에서 지표를 계산하고 낮은 점수에서 확인할 오류와 기본 교정 방향을 배웠습니다.  
02에서는 **실제 오류 선택 → 원문과 규칙 대조 → 한 항목 교정 → 전체 재평가 → 판단** 순서로 직접 적용합니다.

| 순서 | 배우는 것 | 직접 하는 일 |
|---|---|---|
| 1 | 지표별 오류의 원인 | 타입 위반, 인용 불일치, FP와 FN을 원문과 대조하기 |
| 2 | 논리적 모순 후보 | 반대 관계의 출처·조건을 읽고 판단하기 |
| 3 | 실제 교정과 공정한 비교 | 타입 한 건, 인용 한 건을 별도 복사본에서 각각 교정하기 |
| 4 | 전체 지표와 정답 손실 | 각 교정의 효과와 기존 두 추출 버전의 차이를 전체 지표로 확인하기 |
| 5 | 채택 판단 | 좋아진 지표와 나빠진 지표를 함께 보고 근거 남기기 |

**학습 목표**

- 타입 오류, 인용 오류, 의미 오류, 표기 차이와 후처리 손실을 원문과 기록으로 구분할 수 있습니다.  
- 검토한 행만 교정하고 변경한 필드와 원문 근거를 기록할 수 있습니다.  
- 같은 문서·골드·일치 기준으로 변경 전후 전체 결과를 비교할 수 있습니다.  
- 점수 상승과 정답 손실을 함께 확인하고 채택·보류 이유를 설명할 수 있습니다.

**먼저 구분할 용어**

| 용어 | 이 교안에서의 뜻 |
|---|---|
| 추출 행 | 주어·관계·목적어뿐 아니라 타입·evidence·문서 ID까지 담은 저장 항목 |
| 트리플 키 | 평가에서 비교하는 `(주어, 관계, 목적어)` 세 값 |
| 골드 | 평가 전에 확정해 둔 정답 트리플 집합 |
| TP / FP / FN | 골드와 비교해 맞힌 트리플 / 골드에 없는 추출 / 최종 결과에서 놓친 정답 |
| 후처리 검사 | 추출 뒤에 스키마·evidence·문서 절을 확인해 통과 여부를 결정하는 단계 |

따라서 **최종 FN이라고 해서 모델이 트리플을 처음부터 못 뽑았다는 뜻은 아닙니다.**  
처음에는 올바르게 뽑혔지만 evidence 검사에서 행 전체가 제거된 경우도 FN이 됩니다.

**1~5절은 저장 자료로 완주하며 API 키가 필요 없습니다.**  
그 뒤 확장 1은 문맥 처리 실험 3회, 확장 2는 AI 검토 의견 1회를 선택적으로 호출합니다.  
각 호출 셀에는 live_api 태그와 [실제 호출] 주석이 있습니다. 선택하지 않으면 그 네 셀만 건너뛰세요.

<img src="images/quality_workflow.png" width="1000" alt="01에서 만든 지표와 오류 목록을 사용해 수정 방법을 정하고 같은 기준으로 재평가합니다.">

01에서 만든 지표와 오류 목록을 사용해 수정 방법을 정하고 같은 기준으로 재평가합니다.

## 0. 실제 원문과 저장 결과를 준비합니다

교안 01의 실행 변수나 학생 답안을 가져오지 않습니다. 이 준비 셀부터 실행하세요.  
논문 시연은 같은 6편의 기존 추출과 예시 추가 추출을 비교합니다.  
따라하기는 의약품 8문서에서 **실제 근거·절 후처리 검사 전후**를 비교합니다.

의약품 비교에서 01은 스키마와 인용 검사 통과 47관계, 02는 원본의 **절 검사까지 통과한 42관계**를 사용합니다.  
절 검사는 목적어가 해당 효능이나 이상반응 절에 있는지도 확인하므로 두 목록의 FP 수가 다릅니다.

In [ ]:
# 실습에 공통으로 쓸 파일 경로와 읽기, 저장 함수를 준비합니다.

import json
import random
from collections import Counter
from pathlib import Path

data_dir = Path("data")  # 제공된 원문, 추출된 트리플, 골드 파일이 있는 폴더입니다.
output_dir = Path("output")  # 직접 계산한 지표와 검토 기록을 저장할 폴더입니다.
output_dir.mkdir(exist_ok=True)

def load_rows(filename):
    """data 폴더의 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    return [json.loads(line) for line in (data_dir / filename).read_text(encoding="utf-8").splitlines() if line.strip()]

def write_json(filename, value):
    """이번 실습의 결과를 output 폴더에 JSON으로 저장합니다."""
    (output_dir / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_rows(filename, rows):
    """검토 기록을 한 줄에 한 항목인 JSONL로 저장합니다."""
    content = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    (output_dir / filename).write_text(content + "\n", encoding="utf-8")

def triple_key(row):
    """고유 관계를 비교할 (주어, 관계, 목적어) 튜플을 돌려줍니다."""

    # 평가 전에 표기를 바꾸지 않습니다. 이름 정규화는 다음 단원에서 배웁니다.
    return (row["subject"], row["relation"], row["object"])

In [ ]:
# 논문 추출을 관계, 타입 규칙에 따라 통과와 기각으로 나눌 함수를 준비합니다.

signatures = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

def check_signature(row):
    """관계, 타입 위반 사유를 돌려주고, 통과하면 None을 돌려줍니다."""
    if row["relation"] not in signatures:
        return "허용 관계가 아님"
    # signatures의 값은 해당 관계가 요구하는 주어 타입과 목적어 타입입니다.
    subject_type, object_type = signatures[row["relation"]]
    if row["subject_type"] != subject_type:
        return f"주어 타입이 {subject_type} 이어야 함"
    if row["object_type"] != object_type:
        return f"목적어 타입이 {object_type} 이어야 함"

    return None

def split_schema(rows):
    """추출 목록을 스키마 통과 목록과 사유가 붙은 기각 목록으로 나눕니다."""
    valid, rejected = [], []
    for row in rows:
        reason = check_signature(row)
        if reason is None:
            valid.append(row)
        else:
            # 원본은 유지하고 기각 목록에만 사유를 덧붙입니다.
            rejected.append(dict(row, reject_reason=reason))

    return valid, rejected

In [ ]:
# 추출과 골드를 비교해 TP, FP, FN, 정밀도, 재현율, F1을 계산할 함수를 정의합니다.

def measure_exact(rows, gold_rows):
    """고유 관계의 완전일치 TP, FP, FN과 정밀도, 재현율, F1을 돌려줍니다."""
    # 집합으로 바꿔 같은 관계를 여러 번 뽑아도 한 번만 셉니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold_rows}
    tp = len(predicted & expected)  # 추출 결과와 골드 양쪽에 있는 관계입니다.
    fp = len(predicted - expected)  # 골드에 없는 추출입니다. 표기 차이도 포함합니다.
    fn = len(expected - predicted)  # 골드에는 있지만 추출하지 못한 관계입니다.
    # 분모가 없으면 0점 대신 미산출(None)로 남깁니다.
    precision = tp / len(predicted) if predicted else None
    recall = tp / len(expected) if expected else None

    # 골드가 있는데 아무것도 뽑지 않으면 F1은 0입니다. 골드가 없으면 평가에서 별도 표시합니다.
    f1 = 2 * tp / (2 * tp + fp + fn) if expected else None

    return {"predicted": len(predicted), "gold": len(expected), "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# 두 추출 버전과 공통 골드를 준비해 같은 기준으로 비교합니다.

from pprint import pprint

# kg_corpus.jsonl: 논문 발췌 6편의 원문, 문장, 출처입니다. 문서 범위와 인용을 확인합니다.
docs = load_rows("kg_corpus.jsonl")
# doc_ids는 평가 범위, doc_text는 문서 ID로 원문을 찾기 위한 목록입니다.
doc_ids = {doc["doc_id"] for doc in docs}
doc_text = {doc["doc_id"]: doc["text"] for doc in docs}

# gold_triples.jsonl: 같은 논문의 정답 68관계와 출처, 근거입니다. 두 버전의 공통 채점 기준입니다.
gold = load_rows("gold_triples.jsonl")

# triples_raw.jsonl: 프롬프트에 예시를 넣기 전에 추출된 트리플입니다. 비교 기준선입니다.
raw = load_rows("triples_raw.jsonl")

# triples_fewshot.jsonl: 예시를 추가한 뒤 추출된 트리플입니다. 기준선과 비교할 결과입니다.
fewshot = load_rows("triples_fewshot.jsonl")
baseline_valid, _ = split_schema(raw)

paper_inputs = {"원문 문서 수": len(docs),
                "기존 추출 행 수": len(raw),
                "예시 추가 추출 행 수": len(fewshot),
                "골드 트리플 수": len(gold)}

print("[논문 평가 자료]")
pprint(paper_inputs, sort_dicts=False)
print("논문에서 추출한 트리플 한 건:")
pprint(raw[0], sort_dicts=False)

## 1. 지표가 가리킨 오류를 원문에서 확인합니다

### 1-1. 스키마 기각과 인용 불일치를 각각 조사합니다

스키마 준수율이 낮으면 기각 목록을, 근거 원문 일치율이 낮으면 인용 불일치 목록을 확인합니다.  
아래 두 항목은 같은 관계를 두 번 검사하는 것이 아닙니다.  
같은 예시 추가 저장본에서 **서로 다른 두 행**을 골라 두 검사 단계를 비교합니다.

- 사례 1: `type_case`는 관계의 목적어 타입이 스키마 규칙과 달라 기각된 행입니다.  
- 사례 2: `quote_case`는 스키마는 통과했지만 `evidence`가 원문과 문자열로 완전히 같지 않은 행입니다.

In [ ]:
# 타입 오류와 인용 오류를 각각 골라 추출된 필드와 원문을 대조합니다.

review_valid, review_rejected = split_schema(fewshot)

# type_key로 검토할 관계를 특정하고, type_case에 그 기각 행을 가져옵니다.
type_key = ("diphenhydramine", "PALLIATES", "transient insomnia")
type_case = next(row for row in review_rejected if triple_key(row) == type_key)
print("\n[사례 1: 스키마 타입 오류]")
print("타입 오류로 기각된 추출 관계:", type_key)
print("추출된 object_type:", type_case["object_type"])
print("스키마가 요구하는 object_type:", signatures["PALLIATES"][1])
print("대조할 원문 문장:", next(doc for doc in docs if doc["doc_id"] == type_case["source_doc_id"])["sentences"][10])

# 두 번째 사례에서는 CML과 CIP2A 행의 evidence와 출처 문장을 비교합니다.
# repr은 문장부호 차이를 확인하기 위한 출력입니다.
quote_key = ("CML", "ASSOCIATES", "CIP2A")
quote_case = next(row for row in review_valid if triple_key(row) == quote_key)
quote_source = next(doc for doc in docs if doc["doc_id"] == quote_case["source_doc_id"])["sentences"][0]
print("\n[사례 2: evidence 완전일치 실패]")
print("인용을 검토할 추출 관계:", quote_key)
print("추출된 evidence:", repr(quote_case["evidence"]))
print("대조할 원문 문장:", repr(quote_source))
print("추출된 evidence가 원문에 그대로 있음:", quote_case["evidence"] in doc_text[quote_case["source_doc_id"]])

# 마지막 문장부호를 제외한 내용은 같지만, evidence는 원문의 세미콜론을 마침표로 바꾼 기록입니다.
quote_body = quote_case["evidence"][:-1]
print("마지막 문장부호를 제외한 evidence가 원문에 있음:", quote_body in quote_source)
print("추출된 evidence의 끝 문장부호:", repr(quote_case["evidence"][-1]))
print("원문의 같은 위치 문장부호:", repr(quote_source[len(quote_body)]))

| 항목 | 확인한 사실 | 교정할 필드 |
|---|---|---|
| diphenhydramine의 transient insomnia 완화 | 원문이 불면에 사용하는 약으로 명시합니다. 이 스키마는 PALLIATES 목적어를 Disease로 기록하는데 출력은 Symptom입니다. | object_type을 Disease로 교정 |
| CML과 CIP2A의 연관 | 원문은 세미콜론으로 이어지는데 인용문은 앞부분을 마침표로 끝냅니다. 관계를 뒷받침하는 원문 문장을 그대로 대조했습니다. | evidence를 해당 원문 문장으로 교정 |

타입 교정은 **원문의 개체 의미와 이 데이터 모델의 분류 규칙을 대조한 결과**입니다.  
다른 기각 항목까지 같은 타입으로 일괄 변경하지 않습니다. 관계 이름과 개체 이름도 바꾸지 않습니다.

인용 사례는 관계의 의미가 틀렸다는 뜻이 아닙니다. 추출된 `evidence`는 `progression.`으로 끝나지만  
원문은 같은 위치가 `progression;`입니다. 현재 검사는 **원문 문자열을 그대로 인용했는지** 확인하므로 마침표와 세미콜론의 차이도 `False`를 만듭니다.  
두 오류는 원인이 다르므로 3절에서 각각 별도의 복사본을 만들어 한 필드씩 교정합니다.

### 1-2. FP와 FN의 원문을 읽어 의미 오류와 누락을 구분합니다

**FP와 FN은 골드와 다르다는 계산 결과이고, 원인 이름은 아닙니다.**  
FP는 추출에 붙은 근거로, FN은 골드에 기록한 원문으로 돌아갑니다.

| 확인한 차이 | 더 읽을 자료 | 구분할 원인 |
|---|---|---|
| 골드에 없는 추출 FP | 추출 근거 + 골드 정답셋 작성 공통 지침 | 의미를 잘못 읽었는지, 표기만 다른지 |
| 골드에만 있는 관계 FN | 골드 근거 + 원래 추출 + 기각 기록 | 처음부터 안 뽑혔는지, 나중에 기각됐는지 |
| 스키마 기각 | 관계·타입과 기각 사유 | 어떤 규칙을 어겼는지 |

숫자가 같은 두 FP라도 수정 방법은 다를 수 있습니다. 실제 사례를 읽어 봅시다.

In [ ]:
# 기존 논문 추출의 FP, FN을 찾고 대표 사례의 원문 근거를 확인합니다.

gold_keys = {triple_key(row) for row in gold}
baseline_keys = {triple_key(row) for row in baseline_valid}
# 추출에만 있으면 FP, 골드에만 있으면 FN입니다.
baseline_fp = baseline_keys - gold_keys
baseline_fn = gold_keys - baseline_keys
fp_case = next(row for row in baseline_valid
               if triple_key(row) == ("Laquinimod", "TREATS", "multiple sclerosis"))
fn_case = next(row for row in gold
               if triple_key(row) == ("diphenhydramine", "PALLIATES", "transient insomnia"))

print("[대표 FP: 추출에는 있지만 골드에는 없는 트리플]")
print("트리플:", triple_key(fp_case))
print("추출된 evidence:", fp_case["evidence"])

print("\n[대표 FN: 골드에는 있지만 기존 추출에는 없는 트리플]")
print("트리플:", triple_key(fn_case))
print("골드에 기록된 출처와 근거:")
pprint(fn_case["sources"], sort_dicts=False)

In [ ]:
# Laquinimod의 치료 관계가 왜 FP인지 원문과 정답 기준으로 확인합니다.

# gold_guideline.json: 스키마를 정한 뒤 원문 관계를 골드 정답셋에 넣을지 판정하는 지침입니다.
guideline = json.loads((data_dir / "gold_guideline.json").read_text(encoding="utf-8"))

# 이번 후보의 FP 판정에 직접 쓰는 공통 지침만 찾습니다.
possibility_guideline = next(item for item in guideline if item["항목"] == "가능성만 말한 관계")

print("[골드 정답셋 작성 공통 지침]")
print("역할: 스키마·온톨로지를 설계하는 규칙이 아니라, 원문 후보를 골드 정답으로 넣을지 판정하는 규칙입니다.")
print("현재 작업: 골드를 새로 만드는 것이 아니라, 이미 고정된 골드의 판정 근거를 확인합니다.")
print("적용 범위: 특정 샘플이나 TREATS만이 아니라, 가능성만 말한 모든 관계 후보에 적용합니다.")
print("판정 규칙:", possibility_guideline["내용"])

print("\n[이번 Laquinimod TREATS 후보에 지침 적용]")
print("공통 지침에 해당하는 원문 표현:", repr("investigated as"))
for sentence in next(doc for doc in docs if doc["doc_id"] == fp_case["source_doc_id"])["sentences"]:
    if "Laquinimod" in sentence:
        print("후보를 판정할 원문:", sentence)

**Laquinimod의 TREATS가 FP인 이유**

1. 추출 결과에는 `(Laquinimod, TREATS, multiple sclerosis)`가 있습니다.  
2. 원문은 Laquinimod가 치료한다고 보고한 것이 아니라, 치료제로 **연구됐다**고만 말합니다.  
3. 이미 고정된 골드는 `investigated as`처럼 가능성만 말한 표현을 정답 관계로 넣지 않습니다.  
4. 따라서 이 트리플은 추출에만 있고 골드에는 없으므로 FP로 계산됩니다.

지금은 골드 정답셋을 작성하거나 수정하는 단계가 아닙니다.  
**제공된 골드와 그 작성 지침을 읽어, 현재 추출이 왜 FP로 계산됐는지 확인하는 단계입니다.**

**의약품 따라하기의 평가 범위와 골드 포함 기준**

논문 시연 다음에는 의약품 제품 문서에 적용합니다. 여덟 문서의 **[이상반응] 절**에서 `HAS_SIDE_EFFECT` 한 관계만 평가합니다. 주어는 제품명, 목적어는 원문에 이름으로 적힌 증상입니다.

제공된 골드 정답셋을 작성할 때 다음 기준을 적용했습니다.

- **포함**: 가능성 표현이 있어도 [이상반응] 목록에 적힌 증상은 포함합니다.  
- **제외**: 행동 지시나 증상명이 아닌 상태 묘사는 제외합니다.  
- **괄호**: 괄호 안의 부연 증상은 독립된 트리플로 만들지 않습니다.  
- **나열**: 쉼표로 나열된 증상은 각각 분리하되 원문 표기를 유지합니다.

이는 **실습용 골드 정답셋을 구축할 때 적용한 기준**이며, 임상 판단 기준은 아닙니다.

각 트리플에 제품명·관계·증상·문서 ID를 남겨 **어느 문서에서 나온 관계인지 추적**합니다.  
[이상반응] 절이 빈 문서도 평가 대상에 남기고 기대 관계 수를 **0건**으로 기록합니다. 그래야 그 문서를 검토에서 누락하지 않았는지 확인할 수 있습니다.

### 1-3. 의약품 하나의 최종 추출과 골드를 비교합니다

`target_predictions`는 후처리 검사 후 남은 트리플, `target_gold`는 같은 제품의 골드 정답입니다.  
두 목록을 먼저 나란히 보여 주면 **어떤 정답이 최종 결과에서 누락됐는지** 확인할 수 있습니다.  
이 제품의 모든 트리플은 주어와 관계가 같습니다. 반복되는 값은 한 번만 보여 주고, 비교할 **목적어 목록**을 따로 출력합니다.

In [ ]:
# 의약품 실습에 처음 필요한 원문, 최종 추출, 기각 기록, 골드를 이 자리에서 준비합니다.

# fa_corpus.jsonl은 문서 ID로 제품명과 [이상반응] 원문을 다시 찾는 데 사용합니다.
fa_docs = {row["doc_id"]: row for row in load_rows("fa_corpus.jsonl")}

# fa_triples.jsonl에서 이번 평가 범위인 HAS_SIDE_EFFECT만 남깁니다.
fa_all = load_rows("fa_triples.jsonl")
fa = [row for row in fa_all
      if row["relation"] == "HAS_SIDE_EFFECT" and row["source_doc_id"] in fa_docs]

# fa_rejected.jsonl은 트리플이 후처리의 어느 검사에서 제거됐는지 확인하는 기록입니다.
fa_rejected = load_rows("fa_rejected.jsonl")

# fa_gold.jsonl은 최종 추출과 기각 기록을 같은 기준으로 비교할 정답입니다.
fa_gold = load_rows("fa_gold.jsonl")

product_inputs = {"원문 문서 수": len(fa_docs),
                  "후처리 통과 이상반응 트리플 수": len(fa),
                  "기각 기록 수": len(fa_rejected),
                  "골드 이상반응 트리플 수": len(fa_gold)}
print("[의약품 이상반응 평가 자료]")
pprint(product_inputs, sort_dicts=False)

In [ ]:
# 목적어가 현기인 정답 트리플이 evidence 완전일치 검사에서 제거된 과정을 추적할 자료를 준비합니다.

# 하나의 제품만 고정해 비교해야 다른 제품의 트리플이 섞이지 않습니다.
target_doc_id = "drug_199801625"
# 후처리 통과 목록과 골드를 같은 문서 ID로 제한해 비교 범위를 맞춥니다.
target_predictions = [row for row in fa if row["source_doc_id"] == target_doc_id]
target_gold = [row for row in fa_gold if row["source_doc_id"] == target_doc_id]

print("제품:", fa_docs[target_doc_id]["title"])
print("[이상반응]", fa_docs[target_doc_id]["side_effect"])

# 긴 제품명과 관계를 매 행 반복하지 않고, 차이가 나는 목적어를 모아 보여 줍니다.
target_prediction_view = {
    "트리플 수": len(target_predictions),
    "공통 주어": target_predictions[0]["subject"],
    "공통 관계": target_predictions[0]["relation"],
    "목적어 목록": sorted(row["object"] for row in target_predictions),
}
target_gold_view = {
    "트리플 수": len(target_gold),
    "공통 주어": target_gold[0]["subject"],
    "공통 관계": target_gold[0]["relation"],
    "목적어 목록": sorted(row["object"] for row in target_gold),
}
print("\n[target_predictions: 후처리 검사 후 남은 트리플]", len(target_predictions), "건")
pprint(target_prediction_view, sort_dicts=False)
print("\n[target_gold: 같은 제품의 골드 정답 트리플]", len(target_gold), "건")
pprint(target_gold_view, sort_dicts=False)

### 🖐️ 함께 따라하기: 올바른 현기 트리플이 evidence 완전일치 검사에서 제거된 과정을 추적합니다

**목적**

여기서 `현기`는 관계 이름이 아니라 제품 문서의 이상반응 이름, 즉 트리플의 목적어입니다.  
관계 이름은 `HAS_SIDE_EFFECT`이며, 전체 트리플은 "트리부틴정의 이상반응으로 현기가 기록됨"을 뜻합니다.

최종 FN만 보면 모델이 처음부터 트리플을 놓쳤는지, 추출 뒤 검사에서 제거됐는지 알 수 없습니다.  
이 실습은 목적어가 `현기`인 트리플이 어느 단계에서 사라졌는지 추적합니다.

| 확인 단계 | 이 사례에서 확인할 내용 |
|---|---|
| 골드 | 트리플 키가 정답에 있는가 |
| 초기 추출 후보 | 같은 트리플 키가 통과 목록 또는 기각 기록에 있는가 |
| evidence 검사 | 추출된 evidence가 원문과 문자열로 완전히 같은가 |
| 최종 결과 | 후처리 뒤에도 트리플이 남아 있는가 |

**이 사례의 결론**: 주어·관계·목적어는 올바르게 추출됐지만 evidence의 공백이 원문과 달랐습니다.  
그 결과 행 전체가 후처리에서 기각됐고, 최종 평가에서는 FN으로 집계됐습니다.

**할 일**  
- target_gold에는 있고 target_predictions에는 없는 정답 트리플을 **target_missed** 집합으로 만드세요.  
- fa_rejected에서 같은 문서의 `HAS_SIDE_EFFECT` 중 목적어가 `현기`인 기각 트리플을 **rejected_case**로 찾으세요.  
- 기각된 트리플이 초기 추출 후보에 있었는지, 골드와 일치하는지, 최종 결과에 남아 있는지 확인하세요.  
- 기각된 트리플의 evidence와 제품 문서의 [이상반응] 원문을 출력하고 문자열 포함 여부를 확인하세요.  
- **target_note**에 observation(관찰), cause(기록으로 확인한 원인), change(수정할 지시)를 적으세요.

**확인 기준**: 이 제품의 최종 결과에는 목적어가 발진이거나 현기인 두 `HAS_SIDE_EFFECT` 정답 트리플이 없습니다.  
현기는 기각 기록에서 찾을 수 있으므로 초기 추출 뒤 evidence 검사에서 제거된 사례입니다.  
발진은 최종 결과와 기각 기록 모두에 없으므로, 제공된 저장 기록만 보면 초기 추출 단계에서 빠진 사례로 구분합니다.  
두 FN의 원인이 같다고 단정하지 않습니다.

In [ ]:
# 🖐️ 함께 따라하기: 올바른 현기 트리플이 evidence 완전일치 검사에서 제거된 과정을 추적합니다

# 최종 FN을 구한 뒤 fa_rejected를 문서 ID, 관계, 목적어로 좁혀 초기 추출 여부와 기각 원인을 확인하세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

올바른 현기 트리플의 FN에 '증상을 더 많이 뽑아라'라는 지시만 추가하면 해결될까요?

<details><summary>정답 보기</summary>

아닙니다. 주어·관계·목적어는 이미 정답으로 추출됐지만 evidence 문자열이 원문과 달라 기각됐습니다. evidence를 원문 그대로 인용하도록 고친 뒤 같은 검사로 다시 확인해야 합니다.

</details>

## 2. 반대 관계를 찾은 뒤 정말 모순인지 확인합니다

**논리적 모순 후보**는 같은 대상에 양립하기 어려운 주장이 함께 나온 경우입니다.  
예를 들어 같은 약물·유전자에 발현을 높인다는 관계와 낮춘다는 관계가 있으면 검토할 필요가 있습니다.  
다만 **조건과 시점이 다르면 두 진술이 모두 맞을 수 있습니다.**

아래 두 행은 원리를 설명하려고 만든 가상 실험 기록입니다. 실제 의학 연구 결과가 아닙니다.  
UPREGULATES_CG는 발현을 높임, DOWNREGULATES_CG는 발현을 낮춤이라는 뜻입니다.

In [ ]:
# 같은 대상의 반대 발현 관계를 모아 모순 후보를 찾습니다.

# 후보가 실제 모순인지는 원문의 조건을 읽고 판단합니다.
conflict_demo = [
    {"subject": "약물A", "relation": "UPREGULATES_CG", "object": "유전자X",
     "source_doc_id": "실험1", "evidence": "세포A에서 6시간 처리 후 발현이 증가했다."},
    {"subject": "약물A", "relation": "DOWNREGULATES_CG", "object": "유전자X",
     "source_doc_id": "실험2", "evidence": "세포B에서 24시간 처리 후 발현이 감소했다."},
]

def find_conflict_candidates(rows):
    """같은 주어, 목적어의 상반된 발현 관계를 근거와 함께 후보로 돌려줍니다."""
    # groups는 같은 주어와 목적어에 연결된 관계 행을 모은 목록입니다.
    groups = {}
    for row in rows:
        key = (row["subject"], row["object"])
        groups.setdefault(key, []).append(row)
    candidates = []
    for key, items in groups.items():
        relations = {row["relation"] for row in items}
        # 두 반대 관계가 모두 있는 경우만 검토 후보로 남깁니다.
        if {"UPREGULATES_CG", "DOWNREGULATES_CG"} <= relations:
            candidates.append({"pair": key, "rows": items})

    return candidates

for candidate in find_conflict_candidates(conflict_demo):
    print("모순 후보:", candidate["pair"])
    for row in candidate["rows"]:
        print(row["relation"], "/", row["source_doc_id"], "/", row["evidence"])

# 기존 논문 추출에도 동일 검사를 적용합니다. 후보가 0개인 결과도 그대로 표시합니다.
paper_conflicts = find_conflict_candidates(baseline_valid)

print("기존 논문 추출에서 찾은 반대 발현 관계 후보 수:", len(paper_conflicts))

가상 기록의 세포 종류와 시간이 다릅니다. 따라서 반대 관계가 있다는 이유만으로 한쪽을 삭제하지 않습니다.  
검토 기록은 **후보 발견 → 출처·시점·조건 확인 → 모순/조건 차이/판단 보류** 순서로 남깁니다.  
위 자동 검사는 발현 증가·감소라는 한 종류의 후보만 찾습니다. 모든 논리 오류를 찾는 검사는 아닙니다.

**AI 기반 일관성 체크**는 후보·원문·판정 기준을 모델에게 주고 검토 의견을 받는 방법입니다.  
모델 의견도 틀릴 수 있으므로 사람이 최종 판정합니다. 확장 2에 실제 호출 코드가 있습니다.

### ✅ 바로 확인 퀴즈

같은 약물·유전자에 발현 증가와 감소가 함께 있습니다. 자동으로 둘 중 하나를 지워도 될까요?

<details><summary>정답 보기</summary>

안 됩니다. 출처와 실험 조건·시점을 읽어야 합니다. 조건이 다르면 두 진술이 모두 맞을 수 있으므로 먼저 모순 후보로 기록합니다.

</details>

## 3. 원문에서 확인한 오류를 한 항목씩 교정합니다

수정 전후를 비교하려면 **같은 시험으로 채점**해야 합니다.

| 고정할 것 | 이번 논문 비교 |
|---|---|
| 문서 | 같은 논문 발췌 6편 |
| 골드와 관계 범위 | 같은 68관계, 같은 허용 관계 목록 |
| 일치 기준 | 고유한 주어·관계·목적어의 문자열 완전일치 |
| 검사 순서 | 관계·타입 검사 후 고유 관계 비교 |

### 3-1. 타입 교정과 인용 교정을 별도로 실행합니다

1절에서 원문과 규칙을 확인한 두 항목을 수정합니다. 두 실험은 모두 **원래 예시 추가 추출 80행**에서 시작합니다.  
첫 번째 복사본은 타입 한 칸만, 두 번째 복사본은 인용문 한 칸만 바꿉니다.  
이렇게 해야 어떤 변경 때문에 지표가 달라졌는지 알 수 있습니다.

| 실험 | 바꾸는 값 | 바꾸지 않는 것 |
|---|---|---|
| 타입 교정 | diphenhydramine 관계의 object_type: Symptom → Disease | 인용문, 주어, 관계, 목적어 |
| 인용 교정 | CML과 CIP2A 관계의 evidence: 원문 문장 그대로 | 타입, 주어, 관계, 목적어 |

아래 코드는 **이미 저장된 추출 결과에서 사람이 확인한 필드 한 개만 직접 고치는 실험**입니다.  
뒤에서 나오는 지표 변화는 **수동 교정의 효과**이며, 새 프롬프트나 모델의 성능 향상을 뜻하지 않습니다.  
프롬프트 개선 효과를 확인하려면 판정 기준을 지시에 추가한 뒤 새로 추출하고, 그 결과를 같은 골드로 다시 평가해야 합니다.

<img src="images/fair_comparison.png" width="1000" alt="문서·골드·평가 기준을 고정하고 바꾼 요소와 결과를 구분합니다.">

문서·골드·평가 기준을 고정하고 바꾼 요소와 결과를 구분합니다.

In [ ]:
# 원문을 대조해 교정하기로 정한 한 행만 복사본에서 수정하고 변경 기록을 남깁니다.

def correct_reviewed_row(rows, doc_id, key, field, value):
    """문서와 관계로 특정한 한 행의 타입 또는 인용문을 교정한 복사본과 기록을 돌려줍니다."""
    if field not in {"object_type", "evidence"}:
        raise ValueError("이번 실습은 확인한 목적어 타입 또는 인용문만 교정합니다")

    # 문서 ID와 트리플을 함께 비교해 같은 이름이 다른 문서에 나오는 경우를 구분합니다.
    indices = [i for i, row in enumerate(rows)
               if row["source_doc_id"] == doc_id and triple_key(row) == key]
    if len(indices) != 1:
        raise ValueError("교정 대상이 한 행인지 먼저 확인하세요")

    # 각 행도 복사해야 필드를 수정했을 때 원본 딕셔너리가 함께 바뀌지 않습니다.
    corrected = [dict(row) for row in rows]
    index = indices[0]
    previous = corrected[index][field]
    corrected[index][field] = value

    change = {"source_doc_id": doc_id, "triple": list(key), "field": field,
              "before": previous, "after": value, "method": "원문 대조 후 사람 교정"}
    return corrected, change

In [ ]:
# 두 실험은 같은 원본에서 시작하고, 각각 검토한 한 행의 한 필드만 바꿉니다.

type_corrected_rows, type_change = correct_reviewed_row(
    fewshot, type_case["source_doc_id"], type_key, "object_type", "Disease")
print("타입 변경 기록:")
pprint(type_change, sort_dicts=False)

quote_corrected_rows, quote_change = correct_reviewed_row(
    fewshot, quote_case["source_doc_id"], quote_key, "evidence", quote_source)
print("인용 변경 기록:")
pprint(quote_change, sort_dicts=False)

# 수정한 추출과 변경 내역은 output에 저장합니다. data의 원본을 덮어쓰지 않습니다.
write_rows("reviewed_type_triples.jsonl", type_corrected_rows)
write_rows("reviewed_quote_triples.jsonl", quote_corrected_rows)
write_json("reviewed_changes.json", [type_change, quote_change])

### 3-2. 기존 두 추출 버전의 프롬프트 변경도 확인합니다

다음은 방금 실행한 사람 교정과 별개의 비교입니다.  
기존 저장본 두 개는 추출 프롬프트에 관계 판정 예시를 추가하기 전과 후의 결과입니다.  
예시 없이 지시만 주는 방식을 Zero-shot, 소수의 예시도 함께 주는 방식을 Few-shot이라고 합니다.  
예시가 많아졌다는 사실 자체가 개선의 증거는 아닙니다.

아래 문자열은 기존 저장본 생성에 사용한 프롬프트입니다. 새 제안이 아닙니다.  
당시 출력 타입은 저장할 때 그래프 레이블로 옮겼습니다.  
평가용 관계 목록에는 INCLUDES가 있지만 당시 지시는 이를 명시하지 않았다는 차이도 기록해 둡니다.

**자료의 한계**: 예시에 평가 발췌의 개체·관계가 포함됩니다.  
따라서 이번 비교는 개발 자료 안에서 수정 방향을 보는 결과입니다.  
새로운 문서에도 효과가 있는지는 예시에 쓰지 않은 별도 문서로 확인해야 합니다.

In [ ]:
# 두 저장본을 만들 때 사용한 지시와 추가 예시를 확인합니다. 모델 호출은 없습니다.

historical_prompt = '다음 의학 논문 발췌에서 관계 트리플을 뽑아라. 허용 관계는 TREATS(약물->질병, 질병 자체를 조절), PALLIATES(약물->질병, 증상만 눅임), BINDS(약물->유전자, 표적·효소·수송체에 작용), UPREGULATES_CG(약물->유전자, 발현을 높임), DOWNREGULATES_CG(약물->유전자, 발현을 낮춤), ASSOCIATES(질병->유전자), PRESENTS(질병->증상) 뿐이다. 타입은 약물·질병·유전자·증상 중 하나로 적는다. 개체 이름은 원문 표기 그대로 쓰고, 각 트리플에는 근거가 된 원문 구절을 evidence 에 담아라.\n\n발췌: '
historical_examples = '\n예시 1) 원문: "Tapinarof is a topical AHR agonist approved for plaque psoriasis."\n     -> (Tapinarof:약물) -[BINDS]-> (AHR:유전자)\n     -> (Tapinarof:약물) -[TREATS]-> (plaque psoriasis:질병)\n예시 2) 원문: "Actionable CYP2C19 phenotypes were frequent among users of omeprazole."\n     -> (omeprazole:약물) -[BINDS]-> (CYP2C19:유전자)\n개체 이름은 원문에 적힌 표기를 그대로 쓴다.\n'
print('기존 지시:', historical_prompt)
print("표현 설명: 기존 지시의 '눅임'은 당시 저장된 표현이며, 여기서는 '증상 완화'를 뜻합니다.")
print('추가한 예시:', historical_examples)

따라하기에서는 다른 종류의 변경을 확인합니다.  
의약품 저장 자료에는 **근거와 절 검사를 통과한 목록과 기각된 목록**이 모두 있습니다.  
같은 추출의 검사 전 후보를 복원해 후처리 검사 전후를 비교할 수 있습니다.

이것은 **추출 프롬프트 두 버전의 비교가 아니라 후처리 검사의 영향 비교**입니다.  
전체 기각 9건 중 이번 이상반응 관계에 해당하는 6건만 범위에 들어옵니다.  
이 기각은 근거·절 검사 결과이며 스키마 위반으로 세지 않습니다.

In [ ]:
# 통과한 추출에 같은 범위의 기각 행을 더해 후처리 검사 전 목록을 복원합니다.

fa_scoped_rejected = [row for row in fa_rejected
                     if row["source_doc_id"] in fa_docs and row["relation"] == "HAS_SIDE_EFFECT"]
fa_before_postprocess = fa + fa_scoped_rejected

fa_evidence_rejected = [row for row in fa_scoped_rejected
                        if row["reject_reason"] == "근거가 원문에 없음"]
fa_section_rejected = [row for row in fa_scoped_rejected
                       if row["reject_reason"] != "근거가 원문에 없음"]
fa_postprocess_stages = {
    "검사 전 추출 후보": len(fa_before_postprocess),
    "evidence 완전일치 검사 통과": len(fa_before_postprocess) - len(fa_evidence_rejected),
    "근거·절 검사 최종 통과": len(fa),
    "기각: evidence 불일치": len(fa_evidence_rejected),
    "기각: 대상 절 벗어남": len(fa_section_rejected),
    "골드 정답 트리플": len(fa_gold),
}
print("[의약품 HAS_SIDE_EFFECT 후처리 단계]")
pprint(fa_postprocess_stages, sort_dicts=False)

### 🖐️ 함께 따라하기: 프롬프트 수정 계획과 재평가 조건을 적습니다

이 단계에서는 evidence 불일치를 줄이기 위한 **프롬프트 수정 계획**을 세웁니다.  
아직 새 프롬프트로 추출하지 않았으므로 결과 점수는 기록하지 않습니다.

| 키 | 적을 내용 | 필요한 이유 |
|---|---|---|
| `observed_problem` | 원문과 기각 기록에서 확인한 문제 | 수정 대상을 명확히 하기 위함 |
| `prompt_change` | 프롬프트에 추가할 지시 | 확인한 원인과 수정을 연결하기 위함 |
| `evaluation_controls` | 수정 전후에 같게 둘 평가 조건 | 프롬프트 수정 효과만 비교하기 위함 |
| `execution_status` | 새 프롬프트로 재추출했는지 여부 | 미실행 계획을 검증 결과로 오해하지 않기 위함 |

**할 일**: 위 네 키를 **fa_plan**에 담으세요.

**확인 기준**: `evaluation_controls`에는 같은 8문서, `HAS_SIDE_EFFECT`, 같은 골드, 같은 후처리 검사와 완전일치 기준이 있어야 합니다.  
아래에서 계산할 후처리 검사 전후 점수를 이 새 지시의 효과라고 쓰지 않습니다.

In [ ]:
# 🖐️ 함께 따라하기: 프롬프트 수정 계획과 재평가 조건을 적습니다

# 관찰한 문제, 프롬프트에 추가할 지시, 고정할 평가 조건, 실행 상태를 나누어 적으세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

프롬프트와 골드의 표기 규칙을 함께 바꾸고 점수가 올랐습니다. 프롬프트 효과가 확인됐나요?

<details><summary>정답 보기</summary>

채점 기준도 바뀌었으므로 분리해서 판단할 수 없습니다. 같은 골드·일치 기준으로 두 추출을 다시 평가해야 합니다.

</details>

## 4. 전체 지표와 새 오류·정답 손실을 계산합니다

### 4-1. 같은 평가 함수로 교정 전후와 기존 두 버전을 평가합니다

01에서 배운 골드 비교로 두 추출 버전의 차이를 확인합니다.  
**원래 행 → 스키마 통과 행 → 고유 관계 → 같은 골드와 비교** 순서입니다.  
아래 기본 표는 **스키마 통과 추출 전체**의 지표입니다.  
두 검사까지 통과한 결과의 지표는 `after_quote_filter`에 별도로 기록합니다.

두 버전의 사람 채점 표본은 없으므로 논문 샘플 정밀도는 미산출입니다.  
인용 일치율을 샘플 정밀도로 대신 쓰지 않습니다.

In [ ]:
# 두 논문 추출 버전의 전체 품질 지표와 변화량을 같은 함수로 계산합니다.

# 검사 순서와 분모를 고정하고, 0점과 미산출 None을 구분합니다.
def evaluate_version(rows):
    """같은 논문 범위의 스키마, 인용, 중복, 완전일치 지표를 돌려줍니다."""
    scoped = [row for row in rows if row["source_doc_id"] in doc_ids]
    valid, rejected = split_schema(scoped)
    score = measure_exact(valid, gold)
    # verbatim은 비어 있지 않은 근거가 해당 출처 원문에 그대로 있는 행 수입니다.
    verbatim = sum(bool(row["evidence"].strip()) and row["evidence"] in doc_text[row["source_doc_id"]]
                   for row in valid)

    # 최종 사용 후보도 같은 골드로 평가합니다. evidence 검사로 빠진 정답은 골드에 남깁니다.
    candidates = [row for row in valid if row["evidence"].strip()
                  and row["evidence"] in doc_text[row["source_doc_id"]]]

    return {"prediction_stage": "스키마 통과 추출 전체",
            "after_quote_filter": measure_exact(candidates, gold),
            "raw": len(rows), "scoped": len(scoped), "out_of_scope": len(rows) - len(scoped),
            "schema_valid": len(valid), "schema_rejected": len(rejected),
            "schema_rate": len(valid) / len(scoped) if scoped else None,
            "schema_reasons": dict(Counter(row["reject_reason"] for row in rejected)),
            "duplicate_rows": len(valid) - score["predicted"],
            "evidence_verbatim_rows": verbatim,
            "evidence_checked_rows": len(valid),
            "evidence_verbatim_rate": verbatim / len(valid) if valid else None,
            "sample_precision": None, **score}

saved_scores = {"baseline": evaluate_version(raw), "fewshot": evaluate_version(fewshot)}
metric_labels = [("schema_rate", "스키마 준수율"), ("evidence_verbatim_rate", "근거 원문 일치율"),
                 ("duplicate_rows", "중복 행 수"), ("tp", "TP"), ("fp", "FP"), ("fn", "FN"),
                 ("precision", "정밀도"), ("recall", "재현율"), ("f1", "F1")]

def format_metric(value):
    """출력에서 건수는 정수로, 비율은 소수 넷째 자리까지 표시합니다."""
    if value is None:
        return "미산출"
    return f"{value:.4f}" if isinstance(value, float) else str(value)

print("지표 / 기존 / 예시 추가 / 변화(추가 - 기존)")
for key, label in metric_labels:
    before_value, after_value = saved_scores["baseline"][key], saved_scores["fewshot"][key]
    difference = after_value - before_value
    difference_text = f"{difference:+.4f}" if isinstance(difference, float) else f"{difference:+d}"
    print(label, "/", format_metric(before_value), "/", format_metric(after_value), "/", difference_text)

In [ ]:
# 같은 원본에서 한 항목씩 교정한 두 결과를 전체 지표로 다시 평가합니다.

# before는 공통 원본, 나머지 두 값은 서로 독립적으로 교정한 결과입니다.
reviewed_scores = {"before": saved_scores["fewshot"],
                   "type_only": evaluate_version(type_corrected_rows),
                   "quote_only": evaluate_version(quote_corrected_rows)}

reviewed_metric_keys = ["schema_valid", "schema_rejected", "evidence_verbatim_rows",
                        "evidence_checked_rows", "tp", "fp", "fn", "precision", "recall", "f1"]
print("지표 / 교정 전 / 타입 한 건 교정 / 인용 한 건 교정")
for key in reviewed_metric_keys:
    print(key, "/", format_metric(reviewed_scores["before"][key]), "/",
          format_metric(reviewed_scores["type_only"][key]), "/",
          format_metric(reviewed_scores["quote_only"][key]))

reviewed_after_quote = {
    label: result["after_quote_filter"] for label, result in reviewed_scores.items()
}
print("\n[evidence 검사까지 통과한 트리플의 골드 비교]")
pprint(reviewed_after_quote, sort_dicts=False)

| 지표 | 교정 전 | 타입 한 건 교정 | 인용 한 건 교정 |
|---|---:|---:|---:|
| 스키마 준수율 | 65 / 80 = 81.25% | 66 / 80 = 82.50% | 65 / 80 = 81.25% |
| 근거 원문 일치율 | 45 / 65 ≈ 69.23% | 46 / 66 ≈ 69.70% | 46 / 65 ≈ 70.77% |
| TP / FP / FN | 58 / 5 / 10 | 59 / 5 / 9 | 58 / 5 / 10 |
| 정밀도 / 재현율 / F1 | 0.9206 / 0.8529 / 0.8855 | 0.9219 / 0.8676 / 0.8939 | 0.9206 / 0.8529 / 0.8855 |

타입 교정으로 기각됐던 관계 한 건이 통과합니다. 원래 근거가 정확했던 행이 검사 대상에 추가되어  
**인용문을 고치지 않아도 근거 검사의 분자와 분모가 함께 늘어납니다.** 비율만 보지 말고 검사한 행 수를 확인하세요.

인용 교정은 원문 일치 수를 한 건 늘렸지만, 비교하는 주어와 관계, 목적어는 그대로입니다.  
**스키마를 통과한 관계 집합은 같으므로 위 표의 정밀도, 재현율, F1은 그대로입니다.**  
근거 원문 검사까지 통과한 추출에서는 교정한 관계가 다시 포함돼 TP 41에서 42, FN 27에서 26으로 바뀝니다.  
즉, 인용 교정도 **최종 사용 결과의 골드 점수**를 바꿀 수 있습니다.  
두 경우 모두 새 사람 채점표가 없으므로 샘플 정밀도는 미산출입니다.

실제 저장본은 **F1이 약 0.564에서 0.885로 높아졌지만 스키마 준수율은 100%에서 81.25%로 낮아졌습니다.**  
정답을 더 찾는 능력과 형식을 지키는 능력이 함께 좋아진 것은 아닙니다.

비율의 차이는 퍼센트포인트로 읽습니다. 100%에서 81.25%로 바뀌면 **18.75%p 하락**입니다.  
다음 따라하기에서는 의약품 후처리에서도 지표들이 같은 방향으로 움직이는지 확인합니다.

### 🖐️ 함께 따라하기: 실제 근거·절 후처리 검사 전후의 품질 지표를 비교합니다

**할 일**  
- fa_before_postprocess와 fa_gold를 평가한 값을 **fa_before_score**에 담으세요.  
- fa와 같은 fa_gold를 평가한 값을 **fa_after_score**에 담으세요.  
- TP·FP·FN·정밀도·재현율·F1을 두 버전으로 나란히 출력하세요.  
- **fa_metric_change**에 정밀도·재현율·F1의 적용 후 값에서 전 값을 뺀 차이를 담으세요.

**확인 기준**: 검사 전은 48관계 중 TP 41·FP 7·FN 3, 검사 후는 42관계 중 TP 40·FP 2·FN 4입니다.  
정밀도와 재현율의 변화 방향이 같은지 확인하세요.

In [ ]:
# 🖐️ 함께 따라하기: 실제 근거, 절 후처리 검사 전후의 품질 지표를 비교합니다

# 두 추출 목록에 동일한 골드와 measure_exact를 적용한 뒤 값을 빼세요.
# 여기에 코드를 작성하세요.

### 4-2. 총점이 숨기는 사라진 정답을 확인합니다

TP가 늘어도 **예전에 맞혔던 관계 일부를 잃고 다른 정답을 더 많이 찾았을 수 있습니다.**  
집합으로 어느 항목이 바뀌었는지 확인합니다.

| 목록 | 구하는 방법 |
|---|---|
| 새로 맞힌 관계 | 새 추출의 정답 중 기존 추출에 없던 것 |
| 새로 생긴 오답 | 새 추출의 FP 중 기존 추출에 없던 것 |
| 사라진 정답 | 기존 추출의 정답 중 새 추출에 없는 것 |
| 없어진 오답 | 기존 추출의 FP 중 새 추출에 없는 것 |

이전 표본의 항목만 추적하면 새로 생긴 오답을 놓칩니다. 새 결과 **전체**를 비교해야 합니다.

In [ ]:
# 예시 추가 후 생긴 정답, 오답과 사라진 정답, 오답을 관계별로 찾습니다.

# 괄호 안에서 정답, 오답을 고른 뒤 이전/이후 목록과 차이를 구합니다.
few_valid, _ = split_schema(fewshot)
few_keys = {triple_key(row) for row in few_valid}
changes = {
    "new_true_positives": sorted((few_keys & gold_keys) - baseline_keys),
    "new_false_positives": sorted((few_keys - gold_keys) - baseline_keys),
    "lost_true_positives": sorted((baseline_keys & gold_keys) - few_keys),
    "removed_false_positives": sorted((baseline_keys - gold_keys) - few_keys),
    "remaining_false_positives": sorted(few_keys - gold_keys),
    "remaining_false_negatives": sorted(gold_keys - few_keys),
}
change_labels = {"new_true_positives": "새 정답", "new_false_positives": "새 오답",
                 "lost_true_positives": "사라진 정답", "removed_false_positives": "없어진 오답",
                 "remaining_false_positives": "남은 FP", "remaining_false_negatives": "남은 FN"}

for key, items in changes.items():
    print(change_labels[key], len(items), "건")
    pprint(items)

In [ ]:
# 교정으로 새 정답이 생겼는지, 기존 정답이나 오답이 어떻게 달라졌는지 확인합니다.

reviewed_changes = {}
for label, rows in [("type_only", type_corrected_rows), ("quote_only", quote_corrected_rows)]:
    corrected_valid, _ = split_schema(rows)
    corrected_keys = {triple_key(row) for row in corrected_valid}
    reviewed_changes[label] = {
        "new_true_positives": sorted((corrected_keys & gold_keys) - few_keys),
        "new_false_positives": sorted((corrected_keys - gold_keys) - few_keys),
        "lost_true_positives": sorted((few_keys & gold_keys) - corrected_keys),
    }
    print("교정 실험:", label)
    pprint(reviewed_changes[label], sort_dicts=False)

reviewed_comparison = {"method": "원문 대조 후 사람 교정, 두 실험은 같은 원본에서 시작",
                       "versions": reviewed_scores, "changes": reviewed_changes,
                       "edits": [type_change, quote_change]}
write_json("reviewed_comparison.json", reviewed_comparison)

타입 교정에서는 diphenhydramine의 transient insomnia 관계가 새 TP로 들어옵니다.  
새 FP와 기존 정답 손실은 없습니다. 인용 교정에서는 evidence 검사 전 관계 집합은 같지만, 검사 후에는 정답 한 건이 다시 포함됩니다.  
이것은 **검토한 두 행의 교정 효과**입니다. 나머지 타입 오류까지 해결됐거나 추출 모델의 성능이 개선됐다는 뜻은 아닙니다.

### 🖐️ 함께 따라하기: 후처리 검사가 없앤 오답과 정답을 각각 찾습니다

**할 일**  
- fa_before_postprocess, fa, fa_gold를 각각 **fa_before_keys**, **fa_after_keys**, **fa_gold_keys** 집합으로 만드세요.  
- **fa_removed_fp**에 검사로 사라진 오답을, **fa_lost_tp**에 사라진 정답을 담으세요.  
- 두 목록을 출력하고, 사라진 정답이 1절에서 조사한 항목인지 확인하세요.

**확인 기준**: 없어진 오답 5관계, 사라진 정답 1관계입니다.  
F1이 올랐어도 정답 손실이 발생했음을 관계 이름으로 확인합니다.

In [ ]:
# 🖐️ 함께 따라하기: 후처리 검사가 없앤 오답과 정답을 각각 찾습니다

# 기존 오답/기존 정답 집합을 각각 만든 뒤 새 추출을 빼세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

새 버전의 전체 TP가 늘었습니다. 기존 정답이 하나도 사라지지 않았다고 말할 수 있나요?

<details><summary>정답 보기</summary>

아닙니다. 기존 정답을 잃고 더 많은 새 정답을 찾았을 수 있습니다. 기존 TP 집합에서 새 추출 집합을 빼야 손실을 확인할 수 있습니다.

</details>

## 5. 좋아진 점과 나빠진 점을 함께 보고 판단합니다

이번 실습의 결정 규칙은 **F1·재현율이 높아지고 기존 정답 손실이 없으면 다음 개발 실험의 후보로 선택하기**입니다.  
실제 프로젝트에서는 먼저 필요한 품질 조건을 정합니다. 이 규칙이 모든 업무의 정답은 아닙니다.

| 비교 | 좋아진 점 | 남은 문제 | 이번 판단 |
|---|---|---|---|
| 타입 한 건 교정 | 스키마 통과 1행 증가, TP 1 증가, FN 1 감소 | 다른 타입 기각 14행은 별도 검토 필요 | 검토한 타입 교정과 근거를 기록 |
| 인용 한 건 교정 | 원문 일치 1행 증가 | 스키마 통과 추출의 점수는 같고 두 검사 통과 추출의 TP는 1 증가 | 평가 단계를 구분해 보고 |
| 논문 예시 추가 | TP 증가, FP·FN 감소, 기존 정답 손실 없음 | 타입 위반 15행, 평가 자료와 예시가 겹침 | 다음 개발 후보로 선택, 운영 적용 보류 |
| 의약품 근거·절 후처리 검사 | 오답 5관계 감소, 정밀도·F1 상승 | evidence 불일치로 올바른 현기 트리플 1건이 제거되어 재현율 하락 | evidence 인용 지시를 고친 뒤 같은 자료로 재평가 |

**보고서에는 점수·평가 조건·오류 근거·판단 이유를 같이 남깁니다.**  
새로 쓴 프롬프트는 실제 실행 전까지 미실행입니다. 과거 저장본의 점수를 새 제안의 성적으로 쓰지 않습니다.

In [ ]:
# 전체 지표와 정답 손실을 근거로 다음 개발 후보를 고르고 비교 결과를 저장합니다.

before, after = saved_scores["baseline"], saved_scores["fewshot"]
candidate_selected = (after["f1"] > before["f1"] and after["recall"] > before["recall"]
                      and not changes["lost_true_positives"])
decision = "예시 추가 버전을 다음 개발 후보로 선택" if candidate_selected else "채택 보류"
print(decision)

comparison = {"scope": {"documents": sorted(doc_ids), "relations": sorted(signatures),
                        "unit": "고유 (주어, 관계, 목적어)", "matching": "exact"},
              "versions": saved_scores, "changes": changes,
              "reviewed_corrections": reviewed_comparison}
write_json("comparison.json", comparison)

In [ ]:
# 나중에 원문으로 돌아가 검토할 수 있도록 모든 FP, FN의 출처, 근거를 저장합니다.

diagnosis = []
for label, keys in [("FP", baseline_fp), ("FN", baseline_fn)]:
    for key in sorted(keys):
        if label == "FP":
            sources = [{"doc_id": row["source_doc_id"], "evidence": row["evidence"]}
                       for row in baseline_valid if triple_key(row) == key]
        else:
            # FN은 추출 행이 없으므로 정답에 기록된 출처와 근거를 가져옵니다.
            sources = [source for row in gold if triple_key(row) == key for source in row["sources"]]
        diagnosis.append({"subject": key[0], "relation": key[1], "object": key[2],
                          "metric_label": label, "sources": sources, "review_status": "추가 검토 필요"})

write_rows("diagnosis.jsonl", diagnosis)

print("출처를 남긴 FP·FN:", len(diagnosis))

In [ ]:
# 품질 판단 보고서와 다음 실험에 쓸 프롬프트 제안을 저장합니다.

# 새 프롬프트는 미실행이므로 기존 저장본의 점수와 구분합니다.
proposed_rule = "연구 대상으로 삼았다는 진술만으로 TREATS를 만들지 말고, 치료한다고 보고한 원문 근거를 확인하라."
revised_prompt = historical_prompt.replace("\n\n발췌: ", "\n" + proposed_rule + "\n\n발췌: ")
(output_dir / "revised_prompt.txt").write_text(revised_prompt, encoding="utf-8")
report = {
    "evaluation": comparison["scope"], "versions": saved_scores,
    "reviewed_corrections": reviewed_comparison,
    "development_decision": decision, "production_decision": "보류",
    "decision_rule": "F1 상승·재현율 상승·기존 정답 손실 0건",
    "remaining_checks": ["타입 위반 개선", "남은 FP/FN 원문 검토", "예시에 쓰지 않은 문서에서 평가"],
    "new_proposal": {"prompt_file": "revised_prompt.txt", "status": "미실행", "metrics": None},
    "conflict_check": {"rule": "같은 주어·목적어의 반대 발현 관계", "candidates": paper_conflicts},
}

write_json("quality_report.json", report)

print("저장:", output_dir / "quality_report.json")

### 🖐️ 함께 따라하기: 의약품 후처리 검사 비교의 결론을 숫자와 근거로 남깁니다

**할 일**: **fa_report**에 before, after, lost_true_positives, decision, next_check를 담으세요.  
앞에서 구한 실제 두 점수와 사라진 정답을 사용합니다.  
decision에는 정밀도·재현율이 서로 다른 방향으로 변한 사실과 이번 판단을 적으세요.  
next_check에는 올바른 현기 트리플의 evidence 인용 오류를 수정한 후 같은 범위로 재평가한다는 조건을 적으세요.  
write_json으로 my_filter_report.json에 저장하세요.

**확인 기준**: 좋아진 F1만 근거로 쓰지 않고, 정답 1건 손실과 재현율 하락도 설명합니다.

In [ ]:
# 🖐️ 함께 따라하기: 의약품 후처리 검사 비교의 결론을 숫자와 근거로 남깁니다

# 두 점수, 사라진 관계, 판단, 다음 검사를 함께 기록하세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

이제 revised_prompt.txt 파일이 있습니다. 이 새 제안의 효과가 검증됐나요?

<details><summary>정답 보기</summary>

아직 미실행 제안입니다. 실제로 다시 추출하고 같은 기준으로 전체 평가한 결과가 있어야 효과를 말할 수 있습니다.

</details>

## 핵심 실습을 마쳤습니다

- reviewed_comparison.json에는 타입 한 건 교정과 인용 한 건 교정의 전체 지표와 정답 변화가 있습니다.  
- comparison.json에는 같은 기준으로 비교한 두 논문 추출 버전의 점수와 변화 목록도 있습니다.  
- diagnosis.jsonl에는 기준선 FP·FN의 출처와 근거가 있습니다.  
- quality_report.json과 my_filter_report.json에는 좋아진 점·나빠진 점·다음 확인 사항이 있습니다.  
- revised_prompt.txt는 원인에 맞춰 작성한 새 제안이며 아직 실행하지 않았습니다.

| 반드시 구분할 것 | 핵심 해석 |
|---|---|
| 최종 FN과 초기 추출 누락 | FN은 처음부터 못 뽑았거나, 맞게 뽑았지만 후처리에서 제거된 경우를 모두 포함합니다. |
| 점수 상승과 무손실 | 정밀도나 F1이 올라도 기존 TP가 사라졌는지 따로 확인합니다. |
| 저장 행 교정과 모델 개선 | 사람이 한 행을 고친 결과는 새 프롬프트로 재추출한 성능이 아닙니다. |
| AI 검토 의견과 최종 판정 | AI 의견은 검토를 돕는 자료이며, 고정된 기준에 따른 사람의 판정을 대신하지 않습니다. |

다음 확장은 문맥과 AI 의견이 필요한 경우를 실험합니다.  
모델을 호출하지 않아도 지금까지의 비교와 보고서는 완성됩니다.

## 확장 1. 같은 목표 문장에 문맥 처리 세 가지를 적용합니다

**상호참조**는 `These agents` 같은 지시어가 앞 문장의 대상을 가리키는 경우입니다.  
여기서는 목표 문장의 **약물과 H1 receptor 사이 BINDS 세 관계**를 고정해 비교합니다.  
앞 문장을 추가했더라도 앞 문장의 불면 관련 관계는 평가 대상에 넣지 않습니다.

이 절의 세 호출 셀은 **선택 실습**입니다. 세 조건을 모두 실행해야 이 실험의 비교를 마칠 수 있습니다.  
한 조건만 실행했다면 나머지를 미실행으로 기록하세요. 결과는 실행마다 달라질 수 있으며 정답 건수를 미리 제시하지 않습니다.

In [ ]:
# 선택 실습에서 호출할 모델 생성 함수를 정의합니다.

def make_model():
    """.env의 설정으로 구조화 추출에 사용할 모델을 만듭니다."""
    import os
    from dotenv import load_dotenv
    from langchain.chat_models import init_chat_model

    # .env: 사용자가 설정한 API 키와 모델 이름입니다. 함수 호출 시 읽고 값은 출력하지 않습니다.
    load_dotenv(".env")
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(".env에 OPENAI_API_KEY를 설정한 뒤 실제 호출 셀을 실행하세요.")
    model_name = os.getenv("LLM_MODEL", "openai:gpt-5.6-luna")
    return init_chat_model(model_name)

In [ ]:
# 문맥 처리 세 조건에서 공통으로 사용할 트리플 출력 서식을 정의합니다.

from typing import Literal
from pydantic import BaseModel, Field

class ContextTriple(BaseModel):
    """문맥 실험에서 모델이 채울 관계 한 건입니다."""
    subject: str = Field(description="원문에 적힌 주어 이름")
    subject_type: Literal["Compound", "Disease", "Gene", "PharmacologicClass", "Symptom"]
    relation: Literal["TREATS", "PALLIATES", "BINDS", "UPREGULATES_CG", "DOWNREGULATES_CG", "ASSOCIATES", "PRESENTS", "INCLUDES"]
    object: str = Field(description="원문에 적힌 목적어 이름")
    object_type: Literal["Compound", "Disease", "Gene", "PharmacologicClass", "Symptom"]
    evidence: str = Field(description="입력에 제시된 목표 문장에서 근거가 된 구절을 그대로")

class ContextExtraction(BaseModel):
    """목표 문장에서 추출한 관계 목록입니다."""
    triples: list[ContextTriple]

In [ ]:
# 비교할 목표 문장, 앞 문장, 정답을 준비해 문맥 실험의 범위를 고정합니다.

# 골드는 평가에만 사용하고 모델에는 전달하지 않습니다.
context_doc = next(doc for doc in docs if doc["doc_id"] == "PMC13461326")
previous_sentence = context_doc["sentences"][10]
target_sentence = context_doc["sentences"][11]
# 원본 논문 골드의 sent_id는 0부터 시작합니다. 목표 문장의 골드에 기록된 BINDS만 평가합니다.
target_gold = [row for row in gold if row["relation"] == "BINDS" and
               any(source["doc_id"] == context_doc["doc_id"] and source["sent_id"] == 11
                   for source in row["sources"])]

print("앞 문장:", previous_sentence)
print("목표 문장:", target_sentence)
print("평가할 골드 관계 수:", len(target_gold))

# 치환할 이름은 골드에서 복사하지 않고 앞 문장에서 직접 확인한 세 이름입니다.
antecedents = ["diphenhydramine", "doxylamine", "hydroxyzine"]
resolved_sentence = target_sentence.replace("These agents", ", ".join(antecedents))

<img src="images/context_resolution.png" width="1000" alt="목표 문장을 고정한 채 문장만 입력하기, 앞 문장 추가하기, 지시어 치환하기를 비교합니다.">

목표 문장을 고정한 채 문장만 입력하기, 앞 문장 추가하기, 지시어 치환하기를 비교합니다.

In [ ]:
# 문맥 실험에 쓸 공통 프롬프트 함수와 결과 평가 함수를 정의합니다.

def context_prompt(target, context=""):
    """배경 문맥과 추출할 목표 문장을 분리한 프롬프트를 돌려줍니다."""
    return (
        "목표 문장이 말한 약물과 유전자·수용체 사이 BINDS 관계만 뽑아라. "
        "BINDS는 약물이 표적 단백질에 결합하거나 작용하는 관계다. "
        "문맥은 목표 문장의 지시어가 가리키는 이름을 찾는 데만 사용하라. "
        "문맥에만 있는 다른 관계는 뽑지 마라. 이름은 원문 표기를 유지하라.\n"
        f"[배경 문맥]\n{context}\n[목표 문장]\n{target}"
    )

def record_context(reply, label, input_text):
    """목표 관계만 같은 골드로 평가하고 이번 실제 호출 결과를 저장합니다."""
    # 구조화 응답을 기존 검사 함수가 받는 딕셔너리 목록으로 바꿉니다.
    rows = [row.model_dump() for row in reply.triples]
    valid, rejected = split_schema(rows)
    scoped = [row for row in valid if row["relation"] == "BINDS"]

    # 다른 관계는 조용히 없애지 않고 범위 밖 출력으로 따로 셉니다.
    result = {"condition": label, "status": "실행 완료", "input_characters": len(input_text),
              "rows": rows, "schema_rejected": rejected,
              "out_of_scope": [row for row in valid if row["relation"] != "BINDS"],
              "metrics": measure_exact(scoped, target_gold), "scope": "목표 문장의 BINDS"}

    print("문맥 처리 실험 결과:")
    pprint(result, sort_dicts=False)

    write_json(f"context_{label}.json", result)

    return result

**세 조건에서 고정한 것**은 모델 설정·출력 서식·목표 관계·골드·평가 함수입니다.  
바뀌는 것은 입력의 문맥 또는 지시어 표기입니다. 아래 셀 하나가 호출 한 번입니다.  
이 실험에서 `H1 receptor`와 `central H1 receptor`가 다르게 나오면 완전일치에서는 FP와 FN으로 남습니다.  
평가 뒤 원문을 읽어 표기 차이인지 별도로 설명하세요.

In [ ]:
# [실제 호출 1/4] 선택 실습입니다. 목표 문장만 주고 한 번 추출합니다.

sentence_input = context_prompt(target_sentence)
sentence_reply = make_model().with_structured_output(ContextExtraction).invoke(sentence_input)

record_context(sentence_reply, "sentence_only", sentence_input)

In [ ]:
# [실제 호출 2/4] 선택 실습입니다. 목표는 그대로 두고 앞 문장을 추가합니다.

context_input = context_prompt(target_sentence, previous_sentence)
context_reply = make_model().with_structured_output(ContextExtraction).invoke(context_input)

record_context(context_reply, "with_context", context_input)

In [ ]:
# [실제 호출 3/4] 선택 실습입니다. 앞 문장에서 확인한 이름으로 지시어만 치환합니다.

resolved_input = context_prompt(resolved_sentence)
resolved_reply = make_model().with_structured_output(ContextExtraction).invoke(resolved_input)

record_context(resolved_reply, "resolved", resolved_input)

실행했다면 세 출력의 **TP·FP·FN과 범위 밖 결과**를 나란히 읽으세요.  
총 추출 건수가 같아도 목표 BINDS를 맞힌 수는 다를 수 있습니다.  
앞 문맥을 넣어 점수가 달라졌다면 이번 입력에서 관찰한 변화로 기록합니다.  
한 번씩의 실행만으로 어떤 방법이 언제나 더 낫다고 결론 내리지는 않습니다.

실행하지 않았다면 이 실험의 결과는 **미실행**입니다. 위 프롬프트가 결과를 어떻게 바꿀지 예측은 할 수 있지만  
그 예측을 측정값으로 쓰지 않습니다. 뒤의 원문 검토와 보고서 작성은 세 호출을 하지 않아도 진행할 수 있습니다.

### 🖐️ 함께 따라하기: 제품 문서에서도 추출할 절을 분리합니다

**배경**: 제품 설명 전체를 주면 [효능]의 증상까지 이상반응으로 뽑을 수 있습니다.

**요구사항**  
- **fa_target_prompt**에 제품명과 `drug_199801625`의 `side_effect` 원문을 담으세요.  
- **fa_target_prompt**에는 [이상반응] 절에서 `HAS_SIDE_EFFECT`만 추출한다는 지시를 넣으세요.  
- **fa_target_prompt**를 출력만 하세요. 이 따라하기에서는 모델을 호출하지 않습니다.

**확인 기준**: 제품명과 [이상반응] 원문은 있고, [효능]의 본문과 골드 정답 목록은 없습니다.

In [ ]:
# 🖐️ 함께 따라하기: 제품 문서에서도 추출할 절을 분리합니다

# fa_docs에서 해당 제품을 읽고 title과 side_effect만 프롬프트에 넣으세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

가정: 앞 문장을 붙인 조건에서 목표 밖 PALLIATES가 나왔다면, 이번 목표 문장의 재현율 분자에 넣어도 될까요?

<details><summary>정답 보기</summary>

안 됩니다. 이번 평가는 목표 문장의 BINDS만 대상으로 합니다. PALLIATES는 범위 밖 출력으로 남기고, 목표 BINDS의 누락은 그대로 셉니다.

</details>

## 확장 2. 골드 작성 기준 적용에 AI 검토 의견을 활용합니다

**AI 기반 일관성 체크**는 모델에게 후보 관계와 원문을 주고, 골드 정답셋 작성 기준에 맞는지 검토 의견을 받는 방법입니다.  
모델의 의견도 틀릴 수 있으므로 그것을 골드 정답이나 최종 채택 결정으로 바로 바꾸지 않습니다.  
여기서는 `investigated as`가 든 추출 한 건을 골드 정답셋 작성 기준과 함께 검토합니다.

<img src="images/error_diagnosis.png" width="1000" alt="규칙의 표시와 모델의 검토 의견을 원문에 대조한 뒤 사람이 최종 판정합니다.">

규칙의 표시와 모델의 검토 의견을 원문에 대조한 뒤 사람이 최종 판정합니다.

In [ ]:
# AI에 검토를 요청할 관계, 원문, 판정 기준, 답변 서식을 준비합니다.

# 실제 요청은 다음 호출 셀에서 실행합니다.
class ReviewVerdict(BaseModel):
    """원문과 골드 정답셋 작성 기준에 대한 모델의 검토 의견입니다."""
    label: Literal["지지됨", "지지되지 않음", "판단 보류"]
    reason: str = Field(description="원문 표현과 판정 기준을 연결한 설명")

review_candidate = next(row for row in raw if row["subject"] == "Laquinimod"
                        and row["relation"] == "TREATS" and row["object"] == "multiple sclerosis")
review_prompt = (
    "추출 관계를 아래 원문과 골드 정답셋 작성 기준만으로 검토하라. 외부 의학 지식으로 보충하지 마라. "
    "기준: investigated as처럼 연구 대상으로 삼았다는 진술만으로는 TREATS를 정답으로 세우지 않는다. "
    "원문에 치료한다고 보고한 별도 진술이 있는지도 확인하라. 불충분하면 판단 보류를 고르라.\n"
    f"후보: {triple_key(review_candidate)}\n근거: {review_candidate['evidence']}\n"
    f"원문: {doc_text[review_candidate['source_doc_id']]}"
)

print("검토할 관계:", triple_key(review_candidate))
print("추출에 붙은 근거:", review_candidate["evidence"])

아래는 **선택 실습의 마지막 실제 호출 한 번**입니다.  
출력은 모델 의견으로 저장하고 `human_decision`은 `판단 보류`로 남깁니다.  
사람이 원문과 지침을 다시 읽은 다음 최종 기록을 작성해야 합니다.  
실행하지 않았다면 AI 의견도 미실행이며, 이 후보는 원문을 직접 읽어 검토할 수 있습니다.

In [ ]:
# [실제 호출 4/4] 선택 실습입니다. 모델 의견을 사람의 정답으로 자동 채택하지 않습니다.

review_reply = make_model().with_structured_output(ReviewVerdict).invoke(review_prompt)
review_result = {"candidate": review_candidate, "ai_status": "실행 완료",
                 "ai_opinion": review_reply.model_dump(),
                 "human_decision": "판단 보류", "human_reason": "사람이 원문과 기준을 재검토해야 함"}

print("AI 검토 의견과 사람 판정 상태:")
pprint(review_result, sort_dicts=False)

write_json("ai_review.json", review_result)

### 🖐️ 함께 따라하기: 모델 호출 없이 사람의 검토 기록을 남깁니다

**배경**: 제품 문서의 추출 `경련`은 제공 골드에 없습니다. 곧바로 모델 오답이라고 단정하지 않고 골드 정답셋 작성 기준부터 확인합니다.

**요구사항**  
- **fa_review_row**에 `fa` 중 목적어가 `경련`인 한 항목을 담고 해당 문서의 `side_effect`를 출력하세요.  
- **fa_human_review**에 `candidate`, `decision`, `reason`을 담으세요.  
- **fa_human_review**의 `reason`에는 괄호 안 부연 증상을 따로 세는지에 관한 제공 골드 정답셋 작성 기준을 근거로 적으세요.

**확인 기준**: `골드 완전일치 FP`라는 판정과 원문에서 그 표현이 등장하는 맥락을 구분합니다.  
모델을 호출하지 않았으므로 AI가 판정했다고 쓰지 않습니다.

In [ ]:
# 🖐️ 함께 따라하기: 모델 호출 없이 사람의 검토 기록을 남깁니다

# 원문에서 해당 표현이 독립된 증상 이름인지 괄호 안 부연인지 확인하세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

AI가 지지된다고 판정한 관계를 골드에 자동 추가해도 될까요?

<details><summary>정답 보기</summary>

안 됩니다. 모델 의견은 검토 보조 자료입니다. 사람이 원문과 고정된 골드 정답셋 작성 기준으로 판단한 뒤 근거를 남깁니다. 골드를 수정한다면 버전을 올리고 두 추출본을 새 골드로 모두 재평가해야 합니다.

</details>